# Lab 1: Customer Churn Prediction
**MKTS 301 — Machine Learning for Marketing | University of Liverpool**

**Learning objectives:**
- Build a baseline and a trained classifier for churn prediction
- Evaluate models using AUC, precision, recall, and lift
- Interpret results in marketing terms (not just statistical terms)

> **Ground rule:** We always report the baseline first. A model that can't beat random guessing has no marketing value.

In [ ]:
# Cell 1: Imports and seed — always run first
import sys
sys.path.append('../../../')  # access shared utilities

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_curve, auc

from scripts.utils.helpers import set_seed, set_plot_style, describe_target, evaluate_classifier

set_seed(42)
set_plot_style()

print('Environment ready')

In [ ]:
# Cell 2: Load data
# Dataset: Telco Customer Churn (IBM Sample Data)
# Source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# Version: downloaded 2026-01-15
df = pd.read_csv('../data/raw/telco_churn.csv')
df.head()

## Step 1: Understand the target variable

Before building any model, we need to understand the **base rate** — the proportion of customers who actually churned.
This determines:
- whether we have a class imbalance problem
- what a random baseline looks like
- which metrics are most meaningful (AUC vs precision vs lift)

In [ ]:
describe_target(df, 'Churn')

## Step 2: Baseline model

**Always build a baseline before any 'real' model.**
The baseline here is a majority-class classifier — it always predicts 'no churn'.
Any model we build must beat this to be worth deploying.

In [ ]:
# Prepare features (simplified for teaching)
features = ['tenure', 'MonthlyCharges', 'TotalCharges']
target = 'Churn'

df_clean = df[features + [target]].copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)
df_clean = df_clean.dropna()

X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline: always predict majority class
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

print('=== BASELINE ===')
evaluate_classifier(baseline, X_test, y_test)

## Step 3: Logistic Regression

We start with logistic regression because:
1. Coefficients are directly interpretable (log-odds → odds ratios)
2. It gives us a clear baseline for 'simple' ML
3. In marketing, interpretability often matters as much as performance

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(random_state=42)
lr.fit(X_train_sc, y_train)

print('=== LOGISTIC REGRESSION ===')
lr_metrics = evaluate_classifier(lr, X_test_sc, y_test)

# Interpret coefficients
coef_df = pd.DataFrame({
    'feature': features,
    'coefficient': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0])
}).sort_values('odds_ratio', ascending=False)

print('\nCoefficients (standardised):')
print(coef_df.to_string(index=False))
print('\nInterpretation: An odds ratio > 1 increases churn probability.')
print('E.g., odds_ratio=1.5 means a 1SD increase in that feature raises churn odds by 50%.')

## Step 4: Random Forest

Random forest typically outperforms logistic regression on tabular data.
But notice the trade-off: we gain predictive performance and lose direct interpretability.
In practice, you must justify this trade-off to a marketing manager.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print('=== RANDOM FOREST ===')
rf_metrics = evaluate_classifier(rf, X_test, y_test)

## Step 5: ROC curve comparison

The ROC curve shows the precision-recall trade-off across all possible thresholds.
In marketing, we often care about the **top decile** of predictions (who to target in a campaign),
not the full curve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, model, X_t in [
    ('Logistic Regression', lr, X_test_sc),
    ('Random Forest', rf, X_test),
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_t)[:, 1])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (% non-churners targeted)')
ax.set_ylabel('True Positive Rate (% churners caught)')
ax.set_title('Churn Model Comparison — ROC Curve')
ax.legend()
ax.annotate(
    'Top-left = perfect model\nDiagonal = random guessing',
    xy=(0.55, 0.15), fontsize=9, color='gray'
)

plt.tight_layout()
plt.savefig('../../../outputs/figures/lab1_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Model | AUC | Lift | Interpretable? |
|---|---|---|---|
| Baseline | ~0.50 | ~1.0 | N/A |
| Logistic Regression | — | — | ✅ Yes (coefficients) |
| Random Forest | — | — | ⚠️ Partial (feature importance) |

**Key insight:** Lift tells you how much better the model is than random targeting.
A campaign budget that can reach 10% of customers should target those in the top decile of predicted churn probability.

**For your report:** Always state which metric you optimised for and why, given the marketing context.